# Phase 3B (Kaggle): AWQ 4-bit versions

**What is AWQ, in plain words**
AWQ ("activation-aware weight quantization") compresses a model's weights to 4 bits. Rather than shrinking every weight
equally, it first looks at real example inputs to find which weights matter most and protects those. We feed it 256 of our
own banking prompts so it learns what matters *for this task*. The output format is read by **vLLM**, a fast serving
engine for GPUs (used in Phase 4 to measure speed and cost).

**Why two versions.** Qwen3.5 contains a new kind of attention layer (`linear_attn`) that is known to be touchy to
compress. Red Hat's official 4-bit Qwen3.5 build skips it, so we do too, and we try a stricter variant as well:
- **default**: 4-bit for everything in the text model *except* `linear_attn`.
- **mlp_only**: also keep all attention layers at 16-bit; only the MLP blocks are 4-bit (bigger file, safest).
The two run **at the same time, one per GPU**.

**How we score them.** We expand the 4-bit weights back to 16-bit for a quality test with plain transformers. That measures
exactly the accuracy the compressed numbers give. Real *speed* and *memory* are measured later in vLLM on a stronger GPU.

**Before you run:** same three things as 3A (new code zip as dataset version, your Phase 2 adapter dataset attached,
Accelerator = **GPU T4 x2**, Internet **On**). **Time:** about 1.5 hours.

**Heads-up:** this notebook installs `llm-compressor`, which needs specific library versions (transformers 5.15-5.17, torch >= 2.10).
That is why it is a separate notebook from 3A. If the install step reports a torch version conflict, tell Claude.

In [ ]:
import os, glob, shutil, subprocess, sys, json, time, threading

def find_src():
    hits = glob.glob('/kaggle/input/**/src/schema.py', recursive=True)
    if hits:
        return os.path.dirname(os.path.dirname(hits[0]))
    for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):
        shutil.unpack_archive(z, '/kaggle/working/_code')
        hits = glob.glob('/kaggle/working/_code/**/src/schema.py', recursive=True)
        if hits:
            return os.path.dirname(os.path.dirname(hits[0]))
    print(subprocess.run('find /kaggle/input -maxdepth 4 | head -40', shell=True, capture_output=True, text=True).stdout)
    raise FileNotFoundError('code dataset not found: add banking-finetuning-code via Add Input')

SRC_ROOT = find_src()
PROJ = '/kaggle/working/Banking_FInetuning'
shutil.copytree(SRC_ROOT, PROJ, ignore=shutil.ignore_patterns('__pycache__'), dirs_exist_ok=True)
os.chdir(PROJ)
os.makedirs('logs', exist_ok=True)
os.environ['HF_HOME'] = '/kaggle/tmp/hf'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
assert os.path.exists('configs/phase3/fp16.yaml'), 'Old code dataset: upload the newest banking-finetuning-code.zip as a new version'

# Big files (merged model, GGUF, AWQ) go to /kaggle/tmp (lots of space); /kaggle/working is capped at ~20 GB.
os.makedirs('/kaggle/tmp/artifacts', exist_ok=True)
if not os.path.exists('artifacts'):
    os.symlink('/kaggle/tmp/artifacts', 'artifacts')

# The trained adapter from Phase 2: expected as a Kaggle dataset (folder containing adapter_model.safetensors).
ad = glob.glob('/kaggle/input/**/adapter_model.safetensors', recursive=True)
assert ad, 'Add your adapter dataset via Add Input (the Phase 2 adapter folder, containing adapter_model.safetensors)'
ADAPTER_DST = 'outputs/full/qwen3_5_4b/adapter'
shutil.copytree(os.path.dirname(ad[0]), ADAPTER_DST, dirs_exist_ok=True)
print('project at', PROJ, '| adapter from', os.path.dirname(ad[0]), '->', ADAPTER_DST)

### Pinned versions (llm-compressor is strict)
`llmcompressor 0.14.0` (latest stable) requires exactly this family, so we pin all of it:
`transformers==5.17.0`, `accelerate==1.15.0`, `compressed-tensors==0.19.0`, `datasets` 4.8.4-5.0.1, and Kaggle's own `torch`
must be between 2.10 and 2.14. The cell below checks torch first and stops with a clear message if it is out of range.

In [ ]:
import torch
from packaging.version import Version
tv = Version(torch.__version__.split('+')[0])
print('Kaggle torch:', torch.__version__)
assert Version('2.10') <= tv < Version('2.15'), f'torch {tv} is outside the range llmcompressor 0.14.0 supports (2.10-2.14). Tell Claude.'

In [ ]:
!pip list --format=freeze | grep -iE '^(torch|torchvision|torchaudio)==' > /tmp/constraints.txt
!cat /tmp/constraints.txt
!pip install -q -c /tmp/constraints.txt "llmcompressor==0.14.0" "compressed-tensors==0.19.0" "transformers==5.17.0" "accelerate==1.15.0" "datasets>=4.8.4,<=5.0.1" peft scikit-learn pyyaml pandas requests sentencepiece
# Kaggle ships torchao 0.10.0, which the newest peft rejects when merging into a non-quantized model. We do not use torchao.
!pip uninstall -y -q torchao
!pip check 2>&1 | grep -iE 'llmcompressor|compressed|transformers|accelerate|datasets|peft|torch' | head -15 || true

In [ ]:
# Version gate: stop now (not 40 minutes in) if the environment is not exactly what we pinned.
import importlib.metadata as md
want = {'llmcompressor': '0.14.0', 'compressed-tensors': '0.19.0', 'transformers': '5.17.0', 'accelerate': '1.15.0'}
have = {k: md.version(k) for k in want}
print(have)
bad = {k: (have[k], want[k]) for k in want if have[k] != want[k]}
assert not bad, f'Version mismatch (have, want): {bad}. If you re-ran this cell, restart the session first (old versions stay loaded in memory).'
import torch, transformers, peft
print('torch', torch.__version__, '| built for CUDA:', torch.version.cuda, '| cuda available:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())
print('transformers', transformers.__version__, '| peft', peft.__version__)
assert torch.version.cuda is not None and torch.cuda.is_available(), 'No GPU: set Accelerator to GPU T4 x2 and restart the session'
NGPU = torch.cuda.device_count()

In [ ]:
from IPython.display import FileLink, display

def stream(cmd, log_path=None, env=None):
    # Run a command, stream its output into the notebook (and a log file), raise on failure.
    print('$', ' '.join(map(str, cmd)))
    e = dict(os.environ, **(env or {}))
    p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=e)
    log = open(log_path, 'a') if log_path else None
    for line in p.stdout:
        if 'it/s' in line or 's/it' in line:
            continue
        print(line, end='')
        if log:
            log.write(line); log.flush()
    rc = p.wait()
    if rc:
        raise RuntimeError(f'command failed (exit {rc}); see output above')

def save_stage(zip_name, rel_paths):
    # Zip files (paths relative to the project) into /kaggle/working and show a download link.
    here = os.getcwd()
    os.chdir('/kaggle/working')
    try:
        paths = ' '.join(f'Banking_FInetuning/{p}' for p in rel_paths if os.path.exists(f'/kaggle/working/Banking_FInetuning/{p}'))
        subprocess.run(f'rm -f {zip_name}.zip && zip -qr {zip_name}.zip {paths}', shell=True, check=True)
        print(f'{zip_name}.zip = {os.path.getsize(zip_name + ".zip") / 1e6:.1f} MB  <- CLICK TO DOWNLOAD NOW')
        display(FileLink(f'{zip_name}.zip'))
    finally:
        os.chdir(here)

def push_to_hub(local_path, repo_name, path_in_repo='', ignore=None):
    # Optional private upload to YOUR Hugging Face account so big artifacts survive the session and can be
    # pulled into Colab later. Needs a Kaggle Secret named HF_TOKEN (write access). Skips quietly if absent.
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        print('No HF_TOKEN secret found: skipping the Hugging Face upload.')
        return
    from huggingface_hub import HfApi
    api = HfApi(token=token)
    repo_id = f"{api.whoami()['name']}/{repo_name}"
    api.create_repo(repo_id, private=True, exist_ok=True)
    if os.path.isdir(local_path):
        api.upload_folder(folder_path=local_path, repo_id=repo_id, path_in_repo=path_in_repo, ignore_patterns=ignore)
    else:
        api.upload_file(path_or_fileobj=local_path, path_in_repo=path_in_repo or os.path.basename(local_path), repo_id=repo_id)
    print('uploaded (private) ->', f'https://huggingface.co/{repo_id}')

def du(path):
    p = subprocess.run(['du', '-shL', path], capture_output=True, text=True)
    return p.stdout.split()[0] if p.stdout else '?'

def run_jobs(jobs):
    # jobs: {name: (gpu, [(module, [args...]), ...])}. Each job runs in its own thread on its own GPU.
    results = {}
    def worker(name, gpu, steps):
        with open(f'logs/{name}.log', 'w') as log:
            for module, args in steps:
                print('$ python -m', module, *map(str, args), file=log, flush=True)
                p = subprocess.run([sys.executable, '-m', module, *map(str, args)], stdout=log, stderr=subprocess.STDOUT,
                                   env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)))
                if p.returncode:
                    results[name] = f'FAILED in {module} (exit {p.returncode})'
                    return
        results[name] = 'ok'
    def last_lines(name):
        f = f'logs/{name}.log'
        L = [l for l in open(f).read().splitlines() if l.strip() and 'it/s' not in l and 's/it' not in l] if os.path.exists(f) else []
        return ' | '.join(l[:140] for l in L[-2:])
    ts = [threading.Thread(target=worker, args=(n, g, s)) for n, (g, s) in jobs.items()]
    [t.start() for t in ts]
    while any(t.is_alive() for t in ts):
        time.sleep(60)
        for n in jobs:
            print(f'[{n}]', last_lines(n))
        print('---')
    [t.join() for t in ts]
    print(results)
    for n, r in results.items():
        if r != 'ok':
            print(f'===== {n}: {r} - last log lines =====')
            print(subprocess.run(f"grep -v -E 'it/s|s/it' logs/{n}.log | tail -45", shell=True, capture_output=True, text=True).stdout)
    return results

## 1. Data
Training split (for the 256 calibration prompts) and test split (for scoring).

In [ ]:
if not os.path.exists('data/test.jsonl'):
    stream([sys.executable, '-m', 'src.data.prepare'])
print(subprocess.run('wc -l data/train.jsonl data/test.jsonl', shell=True, capture_output=True, text=True).stdout)

## 2. Merge the adapter into the base model
Same step as in 3A (about 5 min).

In [ ]:
stream([sys.executable, '-m', 'src.quantize.merge', '--adapter', ADAPTER_DST, '--out', 'artifacts/merged-fp16'], log_path='logs/merge.log')
print('merged model size:', du('artifacts/merged-fp16'))

## 3. Quantize and score both AWQ variants (in parallel)
For each variant: (1) AWQ-quantize to 4 bits using our calibration prompts, (2) score on the 3,080 test examples.
Progress prints every minute. If your GPU count is 1, set `PARALLEL = False` to run them one after another.

In [ ]:
PARALLEL = NGPU >= 2
variants = {'awq_default': ('default', 'configs/phase3/awq_default.yaml', 'artifacts/awq-default'),
            'awq_mlp_only': ('mlp_only', 'configs/phase3/awq_mlp_only.yaml', 'artifacts/awq-mlp-only')}

def steps(variant, cfg, out):
    return [('src.quantize.awq_llmcompressor', ['--model', 'artifacts/merged-fp16', '--out', out, '--variant', variant]),
            ('src.eval.hf_eval', ['--config', cfg, '--split', 'data/test.jsonl', '--tag', 'finetuned_test', '--batch-size', 16])]

jobs = {name: ((i if PARALLEL else 0), steps(*v)) for i, (name, v) in enumerate(variants.items())}
if PARALLEL:
    results = run_jobs(jobs)
else:
    results = {}
    for name, job in jobs.items():
        results.update(run_jobs({name: job}))

## 4. Results
Sizes of the compressed models and their scores. The full comparison table (with fp16 and GGUF) is built after you bring the files from 3A and 3B together.

In [ ]:
# Write the results to results/phase3/ (like 3A does) so the numbers AND the model sizes are saved, not just printed.
os.makedirs('results/phase3', exist_ok=True)
KEYS = ('intent_accuracy', 'intent_macro_f1', 'invented_label_rate', 'json_valid_rate', 'urgency_accuracy', 'needs_human_accuracy')
# fp16 reference measured in notebook 3A (same 3,080 test examples)
REF = {'size': 9.10, 'intent_accuracy': 0.9373, 'intent_macro_f1': 0.9375, 'invented_label_rate': 0.0006,
       'json_valid_rate': 1.0, 'urgency_accuracy': 0.9831, 'needs_human_accuracy': 0.9912}
sizes, table = {}, []
for name, (_, cfg_path, out) in variants.items():
    nbytes = int(subprocess.run(['du', '-sbL', out], capture_output=True, text=True).stdout.split()[0])
    sizes[name] = round(nbytes / 1e9, 2)
    r = json.load(open(f'outputs/phase3/{name}/finetuned_test.json'))
    table.append((name, sizes[name], r))
    print(f'{name:<14} size on disk: {sizes[name]} GB', {k: round(v, 4) for k, v in r.items() if k in KEYS})
head = '| variant | size (GB) | intent acc | delta vs fp16 | macro-F1 | invented labels | valid JSON | urgency | needs_human |'
lines = [head, '|' + '---|' * 9,
         f"| fp16 merged (from 3A) | {REF['size']:.2f} | {REF['intent_accuracy']:.4f} | +0.0000 | {REF['intent_macro_f1']:.4f} | {REF['invented_label_rate']:.2%} | {REF['json_valid_rate']:.2%} | {REF['urgency_accuracy']:.4f} | {REF['needs_human_accuracy']:.4f} |"]
for name, sz, r in table:
    lines.append(f"| {name} | {sz:.2f} | {r['intent_accuracy']:.4f} | {r['intent_accuracy'] - REF['intent_accuracy']:+.4f} | {r['intent_macro_f1']:.4f} | {r['invented_label_rate']:.2%} | {r['json_valid_rate']:.2%} | {r['urgency_accuracy']:.4f} | {r['needs_human_accuracy']:.4f} |")
md = chr(10).join(lines) + chr(10)
open('results/phase3/awq_summary.md', 'w').write(md)
json.dump(sizes, open('results/phase3/awq_sizes.json', 'w'))
print(md)

## 5. Save now
Download the results zip. The optional upload puts the two AWQ models into a private Hugging Face repo for Phase 4.

In [ ]:
save_stage('phase3b_results', ['outputs/phase3', 'results/phase3', 'logs'])

In [ ]:
push_to_hub('artifacts/awq-default', 'banking-triage-qwen3_5_4b-quantized', path_in_repo='awq-default')
push_to_hub('artifacts/awq-mlp-only', 'banking-triage-qwen3_5_4b-quantized', path_in_repo='awq-mlp-only')